# Step E — DL Top-1% Anchor, Final SVD Polarity & Comparison with Raw DL
Re-implements `E_Po_all_15OBS.m`, replacing the manual anchor (step D) with  
**DL picks whose confidence is in the top 1% for that station**.  

For each station:  
1. Find the top-1% confidence threshold → printed for reference  
2. Compute `WTcatpol = Σ(DL_pol × U_val) / Σ|U_val|` over anchor events  
3. Assign `SVD_final = sign(U × sign(WTcatpol))` if `|U| > threshold`, else 0  
4. Compare SVD_final vs raw DL polarity event-by-event  
5. Plot top-20 waveforms + polarity markers per station  
6. Save to `03-output-graphics/cc_svd_validation/`

In [1]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import h5py
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

DATA   = Path('/Users/mczhang/Documents/GitHub/FM7/02-data')
OUT_C  = DATA / 'C_SVD_py'
OUT_E  = DATA / 'E_Po_py'; OUT_E.mkdir(exist_ok=True)
WAVE_H5   = DATA / 'A_ID'  / 'waves_Y1_ge8.h5'
DL_H5     = DATA / 'E_Po'  / 'dl_polarity_Y1_ge8.h5'
FIG_DIR   = Path('/Users/mczhang/Documents/GitHub/FM7/03-output-graphics/cc_svd_validation')
FIG_DIR.mkdir(parents=True, exist_ok=True)
YEAR = 'Y1'
FS   = 200
T    = np.arange(264) / FS - 0.32   # time axis relative to P pick

DL_CONF_MIN = 0.8   # minimum DL confidence for agreement comparison
print('=== Top-3% DL Confidence Thresholds (for reference) ===')


=== Top-3% DL Confidence Thresholds (for reference) ===


In [2]:
# Pre-load DL data
with h5py.File(DL_H5) as f:
    DL_EIDS = f['event_ids'][:]
    DL_STAS = sorted(f['polarity'].keys())
    DL_POL  = {s: f[f'polarity/{s}'][:].astype(int)   for s in DL_STAS}
    DL_CONF = {s: f[f'confidence/{s}'][:].astype(float) for s in DL_STAS}

with h5py.File(WAVE_H5) as f:
    WAVE_EIDS = f['event_ids'][:]
    WAVE_STAS = sorted(f['waveforms'].keys())

# Build eid → index maps
dl_eid2idx   = {int(e): i for i, e in enumerate(DL_EIDS)}
wave_eid2idx = {int(e): i for i, e in enumerate(WAVE_EIDS)}


In [3]:
# ── Main processing ─────────────────────────────────────────
SUMMARY = []
N_TOTAL_EVENTS = len(DL_EIDS)   # total events in dataset

for C_path in sorted(OUT_C.glob(f'C_{YEAR}_*.npy')):
    sta = C_path.stem.replace(f'C_{YEAR}_', '')
    svd_result = np.load(C_path)   # (N, 6): eid, u_val, svd_pol, agree, disagree, thr_log

    if sta not in DL_POL:
        print(f'{sta}: no DL data, skip')
        continue

    # Align DL data to SVD events
    svd_eids = svd_result[:, 0].astype(int)
    N = len(svd_eids)

    dl_pol  = np.zeros(N, dtype=int)
    dl_conf = np.zeros(N)
    for k, eid in enumerate(svd_eids):
        if eid in dl_eid2idx:
            idx = dl_eid2idx[eid]
            dl_pol[k]  = DL_POL[sta][idx]
            dl_conf[k] = DL_CONF[sta][idx]

    # ── Top-1% confidence threshold ────────────────────────
    has_pick = dl_pol != 0
    if has_pick.sum() < 5:
        print(f'{sta}: too few DL picks ({has_pick.sum()}), skip')
        continue
    conf_vals = dl_conf[has_pick]
    top1_thr  = np.percentile(conf_vals, 97)
    n_top3    = (conf_vals >= top1_thr).sum()
    print(f'{sta:5s}: top-3% conf threshold = {top1_thr:.6f}  (n_anchors = {n_top3})')

    # ── WTcatpol: anchor with top-1% DL picks ──────────────
    anchor_mask = has_pick & (dl_conf >= top1_thr)
    if anchor_mask.sum() == 0:
        print(f'  {sta}: no anchor events above top-1% threshold, skip')
        continue

    u_vals    = svd_result[:, 1]
    thr_log   = svd_result[0, 5]
    n_matrix  = len(svd_eids)   # events in SVD matrix
    WTcatpol = (np.sum(dl_pol[anchor_mask] * u_vals[anchor_mask]) /
                np.sum(np.abs(u_vals[anchor_mask])))

    # ── Final SVD polarity ──────────────────────────────────
    thr_val = 10 ** thr_log
    svd_final = np.sign(u_vals * np.sign(WTcatpol))
    svd_final[np.abs(u_vals) < thr_val] = 0   # below threshold → no pick

    # ── Comparison (all DL + high-conf DL >= 0.8) ──────────
    dl_hq     = has_pick & (dl_conf >= DL_CONF_MIN)
    both_all  = (svd_final != 0) & has_pick
    both_hq   = (svd_final != 0) & dl_hq
    agree_all = both_all & (svd_final == dl_pol)
    agree_hq  = both_hq  & (svd_final == dl_pol)
    n_svd     = (svd_final != 0).sum()
    n_dl      = has_pick.sum()
    n_dl_hq   = dl_hq.sum()
    n_both    = both_all.sum()
    n_both_hq = both_hq.sum()
    n_agree   = agree_all.sum()
    n_agree_hq= agree_hq.sum()
    agr_rate  = n_agree    / max(n_both,    1)
    agr_hq    = n_agree_hq / max(n_both_hq, 1)

    cc_det_pct  = round(n_svd / N_TOTAL_EVENTS * 100, 1)
    pct_in_svd   = round(n_svd / max(n_matrix, 1) * 100, 1)
    SUMMARY.append(dict(
        station=sta, WTcatpol=round(WTcatpol, 4),
        top3_thr=round(top1_thr, 6), n_anchors=n_top3,
        n_matrix=n_matrix, n_svd=n_svd,
        pct_in_svd=pct_in_svd, cc_det_pct=cc_det_pct,
        svd_thr=f'1e{thr_log:.0f}',
        n_dl=n_dl, n_dl_hq=n_dl_hq,
        n_both=n_both, n_agree=n_agree, agree_pct=round(agr_rate*100, 1),
        n_both_hq=n_both_hq, n_agree_hq=n_agree_hq, agree_hq_pct=round(agr_hq*100, 1)
    ))

    # Save per-station results
    out_arr = np.column_stack([svd_eids, u_vals, svd_final, dl_pol, dl_conf,
                               np.full(N, WTcatpol), np.full(N, top1_thr)])
    np.save(OUT_E / f'E_{YEAR}_{sta}.npy', out_arr)

print()
df_sum = pd.DataFrame(SUMMARY)
if len(df_sum):
    cols = ['station','n_matrix','n_svd','pct_in_svd','cc_det_pct','svd_thr',
            'n_dl','n_agree','agree_pct',
            'n_dl_hq','n_both_hq','n_agree_hq','agree_hq_pct']
    print('Comparison: all DL picks vs DL conf >= 0.8')
    print(df_sum[cols].to_string(index=False))
    df_sum.to_csv(OUT_E / f'E_{YEAR}_summary.csv', index=False)


01A  : top-3% conf threshold = 0.999877  (n_anchors = 16)
02A  : top-3% conf threshold = 0.999989  (n_anchors = 18)
03A  : top-3% conf threshold = 0.999984  (n_anchors = 30)
04A  : top-3% conf threshold = 0.999963  (n_anchors = 24)
05A  : top-3% conf threshold = 0.999946  (n_anchors = 35)
06A  : top-3% conf threshold = 0.999975  (n_anchors = 33)
07A  : top-3% conf threshold = 0.999996  (n_anchors = 38)
08A  : top-3% conf threshold = 0.999991  (n_anchors = 46)
09A  : top-3% conf threshold = 0.999993  (n_anchors = 23)
10A  : top-3% conf threshold = 0.999971  (n_anchors = 49)
11A  : top-3% conf threshold = 0.999993  (n_anchors = 31)
12A  : top-3% conf threshold = 0.999949  (n_anchors = 27)
13A  : top-3% conf threshold = 0.999961  (n_anchors = 21)
14A  : top-3% conf threshold = 0.999983  (n_anchors = 27)
AS1  : top-3% conf threshold = 0.999993  (n_anchors = 37)
AS2  : top-3% conf threshold = 0.999978  (n_anchors = 28)
CC1  : top-3% conf threshold = 0.999966  (n_anchors = 56)
EC1  : top-3% 

In [4]:
# ── Top-20 waveform + polarity plots ────────────────────────
with h5py.File(WAVE_H5) as f:
    WAVES_ALL = {s: f[f'waveforms/{s}'][:] for s in WAVE_STAS}

POL_COLORS = {1: '#d62728', -1: '#2166ac', 0: 'gray'}
MARKER_S   = {1: '^', -1: 'v', 0: 'o'}
MARKER_SZ  = 120

for E_path in sorted(OUT_E.glob(f'E_{YEAR}_*.npy')):
    sta = E_path.stem.replace(f'E_{YEAR}_', '')
    arr = np.load(E_path)
    # cols: eid, u_val, svd_final, dl_pol, dl_conf, WTcatpol, top1_thr
    eids_s    = arr[:, 0].astype(int)
    u_vals    = arr[:, 1]
    svd_final = arr[:, 2].astype(int)
    dl_pol    = arr[:, 3].astype(int)
    dl_conf   = arr[:, 4]

    # Select 20 random anchor events (top-1% DL conf — these set WTcatpol)
    top1_thr  = arr[0, 6]
    anchor_idx = np.where(dl_conf >= top1_thr)[0]
    rng = np.random.default_rng(seed=42)
    if len(anchor_idx) > 20:
        top_idx = rng.choice(anchor_idx, size=20, replace=False)
        top_idx = top_idx[np.argsort(-np.abs(u_vals[top_idx]))]  # sort by |u| for display
    else:
        top_idx = anchor_idx[np.argsort(-np.abs(u_vals[anchor_idx]))]
    n_show = len(top_idx)

    fig, axes = plt.subplots(n_show, 1, figsize=(12, 2*n_show), facecolor='white')
    if n_show == 1:
        axes = [axes]
    fig.suptitle(f'Station {sta} | {n_show} random anchor waveforms (top-1% DL conf) | Y1'
                 f' -- Red=Up Blue=Down (SVD filled / DL open)',
                 fontsize=11, fontweight='bold', y=1.001)

    for row, k in enumerate(top_idx):
        ax = axes[row]
        eid = eids_s[k]

        # Load waveform
        wave = None
        if sta in WAVES_ALL and eid in wave_eid2idx:
            w = WAVES_ALL[sta][wave_eid2idx[eid]]
            if np.isfinite(w).all():
                wave = w

        if wave is not None:
            peak = np.max(np.abs(wave)) or 1.0
            ax.plot(T, wave / peak, color='#555555', lw=0.8, zorder=1)
        else:
            ax.text(0, 0, 'no waveform', ha='center', va='center',
                    color='gray', fontsize=8)

        # Shade CC windows
        ax.axvspan(T[44], T[103], color='lightyellow', alpha=0.5, zorder=0)
        ax.axvspan(T[44], T[143], color='lightyellow', alpha=0.25, zorder=0)
        ax.axvline(0, color='k', lw=0.7, ls='--', zorder=2)

        # SVD polarity marker (filled)
        sp = svd_final[k]
        if sp != 0:
            ax.scatter([0.06], [0.6 * sp], s=MARKER_SZ,
                       marker=MARKER_S[sp], color=POL_COLORS[sp],
                       zorder=5, label=f'SVD={sp:+d}')

        # DL polarity marker (open)
        dp = dl_pol[k]
        if dp != 0:
            ax.scatter([0.12], [0.6 * dp], s=MARKER_SZ,
                       marker=MARKER_S[dp], facecolors='none',
                       edgecolors=POL_COLORS[dp], linewidths=1.5,
                       zorder=5, label=f'DL={dp:+d}(c={dl_conf[k]:.2f})')

        # Agree / disagree label
        match = (sp == dp and sp != 0)
        lbl = '✓' if match else ('✗' if sp != 0 and dp != 0 else '?')
        lbl_col = 'green' if match else 'red'
        ax.set_xlim(-0.15, 0.6)
        ax.set_ylim(-1.4, 1.4)
        ax.set_yticks([])
        title = (f'eid={eid}  |U|={abs(u_vals[k]):.4f}  '
                 f'SVD={sp:+d}  DL={dp:+d}(c={dl_conf[k]:.2f})  {lbl}')
        ax.set_title(title, fontsize=7.5, color=lbl_col if sp != 0 and dp != 0 else '#333333',
                     pad=1)
        ax.tick_params(labelsize=6)
        if row == n_show - 1:
            ax.set_xlabel('Time relative to P pick (s)', fontsize=8)

    plt.tight_layout()
    out_fig = FIG_DIR / f'anchor20_{YEAR}_{sta}.png'
    fig.savefig(out_fig, dpi=100, bbox_inches='tight')
    plt.close(fig)
    print(f'  {sta}: {n_show} anchors plotted → {out_fig.name}')

print('\nStep E done. Figures in:', FIG_DIR)

  01A: 16 anchors plotted → anchor20_Y1_01A.png


  02A: 18 anchors plotted → anchor20_Y1_02A.png


  03A: 20 anchors plotted → anchor20_Y1_03A.png


  04A: 20 anchors plotted → anchor20_Y1_04A.png


  05A: 20 anchors plotted → anchor20_Y1_05A.png


...

  CC1: 20 anchors plotted → anchor20_Y1_CC1.png


  EC1: 20 anchors plotted → anchor20_Y1_EC1.png


  EC2: 20 anchors plotted → anchor20_Y1_EC2.png


  EC3: 20 anchors plotted → anchor20_Y1_EC3.png


  ID1: 20 anchors plotted → anchor20_Y1_ID1.png

Step E done. Figures in: /Users/mczhang/Documents/GitHub/FM7/03-output-graphics/cc_svd_validation


In [5]:
# ── Summary plot: agreement rate by station ──────────────────
df = pd.read_csv(OUT_E / f'E_{YEAR}_summary.csv')
df = df.sort_values('agree_pct', ascending=True)

fig, ax = plt.subplots(figsize=(10, 5), facecolor='white')
colors = ['#2196f3' if 'A' in s or 'B' in s or any(c.isdigit() for c in s[:3])
          else '#4caf50' for s in df['station']]
bars = ax.barh(df['station'], df['agree_pct'], color=colors, alpha=0.85, edgecolor='none')
ax.axvline(90, color='gray', ls='--', lw=1, label='90%')
ax.axvline(95, color='red',  ls='--', lw=1, label='95%')
for bar, row in zip(bars, df.itertuples()):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{row.agree_pct:.1f}%  (n={row.n_both})',
            va='center', fontsize=8)
ax.set_xlabel('SVD–DL Agreement Rate (%)', fontsize=10)
ax.set_title(f'SVD-CC vs Raw DL Polarity Agreement by Station  |  Y1', fontsize=11)
ax.legend(fontsize=9)
ax.set_xlim(0, 110)
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
out_sum = FIG_DIR / f'agreement_summary_{YEAR}.png'
fig.savefig(out_sum, dpi=150, bbox_inches='tight')
plt.close(fig)
print(f'Summary figure → {out_sum}')
df


Summary figure → /Users/mczhang/Documents/GitHub/FM7/03-output-graphics/cc_svd_validation/agreement_summary_Y1.png


,station,WTcatpol,top3_thr,n_anchors,n_matrix,n_svd,pct_in_svd,cc_det_pct,svd_thr,n_dl,n_dl_hq,n_both,n_agree,agree_pct,n_both_hq,n_agree_hq,agree_hq_pct
16,CC1,0.4954,0.999966,56,1842,1615,87.7,52.0,1.000000e-08,1842,1112,1615,578,35.8,929,210,22.6
3,04A,-0.9725,0.999963,24,772,59,7.6,1.9,1.000000e-16,772,551,59,22,37.3,23,7,30.4
18,EC2,0.2406,0.999905,51,1687,1410,83.6,45.4,1.000000e-09,1687,1050,1410,630,44.7,856,332,38.8
9,10A,0.4448,0.999971,49,1606,1358,84.6,43.7,1.000000e-05,1606,1121,1358,613,45.1,976,417,42.7
15,AS2,-0.9540,0.999978,28,909,704,77.4,22.7,1.000000e-10,909,667,704,348,49.4,529,267,50.5
12,13A,-0.8622,0.999961,21,671,509,75.9,16.4,1.000000e-04,671,489,509,299,58.7,359,219,61.0
5,06A,-0.9993,0.999975,33,1082,793,73.3,25.5,1.000000e-12,1082,882,793,468,59.0,678,411,60.6
13,14A,0.8144,0.999983,27,895,703,78.5,22.6,1.000000e-05,895,676,703,449,63.9,541,368,68.0
11,12A,0.2643,0.999949,27,884,625,70.7,20.1,1.000000e-06,884,632,625,407,65.1,457,317,69.4
0,01A,-1.0000,0.999877,16,533,466,87.4,15.0,1.000000e-04,533,357,466,312,67.0,324,234,72.2


In [6]:
# ── Print summary table as PDF ───────────────────────────────────────────────
import matplotlib.backends.backend_pdf as pdf_backend
from matplotlib.patches import FancyBboxPatch

df = pd.read_csv(OUT_E / f'E_{YEAR}_summary.csv')
cols_show = ['station','n_matrix','n_svd','pct_in_svd','cc_det_pct','svd_thr',
             'n_dl','n_agree','agree_pct',
             'n_dl_hq','n_both_hq','n_agree_hq','agree_hq_pct']
col_labels = ['Station','N in SVD','N polarity','% in SVD','CC det%','SVD thr',
              '#DL all','#Agree all','Agree%(all)',
              '#DL>=0.8','#Both>=0.8','#Agree>=0.8','Agree%(>=0.8)']

data = df[cols_show].values.tolist()

fig, ax = plt.subplots(figsize=(18, 0.45 * len(data) + 1.5))
ax.axis('off')

tbl = ax.table(
    cellText=data,
    colLabels=col_labels,
    cellLoc='center', loc='center'
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(8)
tbl.scale(1, 1.6)

# Header style
for j in range(len(col_labels)):
    tbl[(0, j)].set_facecolor('#2c3e50')
    tbl[(0, j)].set_text_props(color='white', fontweight='bold')

# Row alternating colours; highlight low agree% in red, high in green
for i, row in enumerate(data, start=1):
    agree = float(row[-1])   # agree_hq_pct
    wt    = abs(float(row[1])) if row[1] != '' else 0   # top3_thr is float
    for j in range(len(col_labels)):
        cell = tbl[(i, j)]
        cell.set_facecolor('#f2f2f2' if i % 2 == 0 else 'white')
    # Agree% cell colour
    agree_cell = tbl[(i, len(col_labels) - 1)]   # last col = Agree%(>=0.8)
    if agree >= 75:
        agree_cell.set_facecolor('#c8e6c9')
    elif agree < 50:
        agree_cell.set_facecolor('#ffcdd2')
    else:
        agree_cell.set_facecolor('#fff9c4')

ax.set_title(f'CC-SVD vs DL Polarity Comparison — Y1 (N=3105 events)\n'
             f'Top-3% DL anchor | left=all DL  right=DL conf>=0.8',
             fontsize=11, fontweight='bold', pad=12)

plt.tight_layout()
out_pdf = FIG_DIR / f'E_{YEAR}_summary_table.pdf'
out_png = FIG_DIR / f'E_{YEAR}_summary_table.png'
fig.savefig(out_pdf, dpi=150, bbox_inches='tight')
fig.savefig(out_png, dpi=150, bbox_inches='tight')
plt.close(fig)
print(f'Saved → {out_pdf}')
print(f'Saved → {out_png}')
df[cols_show]


Saved → /Users/mczhang/Documents/GitHub/FM7/03-output-graphics/cc_svd_validation/E_Y1_summary_table.pdf
Saved → /Users/mczhang/Documents/GitHub/FM7/03-output-graphics/cc_svd_validation/E_Y1_summary_table.png


,station,n_matrix,n_svd,pct_in_svd,cc_det_pct,svd_thr,n_dl,n_agree,agree_pct,n_dl_hq,n_both_hq,n_agree_hq,agree_hq_pct
0,01A,533,466,87.4,15.0,1.000000e-04,533,312,67.0,357,324,234,72.2
1,02A,593,447,75.4,14.4,1.000000e-07,593,321,71.8,487,377,289,76.7
2,03A,996,898,90.2,28.9,1.000000e-06,996,703,78.3,773,707,594,84.0
3,04A,772,59,7.6,1.9,1.000000e-16,772,22,37.3,551,23,7,30.4
4,05A,1135,973,85.7,31.3,1.000000e-09,1135,718,73.8,847,745,581,78.0
5,06A,1082,793,73.3,25.5,1.000000e-12,1082,468,59.0,882,678,411,60.6
6,07A,1243,946,76.1,30.5,1.000000e-07,1243,774,81.8,1035,836,722,86.4
7,08A,1511,1271,84.1,40.9,1.000000e-07,1511,1006,79.2,1224,1068,877,82.1
8,09A,765,674,88.1,21.7,1.000000e-07,765,477,70.8,518,465,360,77.4
9,10A,1606,1358,84.6,43.7,1.000000e-05,1606,613,45.1,1121,976,417,42.7
